In [1]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import camelot as cam
import re
from pdfminer.high_level import extract_text
from PyPDF2 import PdfReader
from datetime import datetime
import tempfile
from pdfminer.pdfparser import PDFSyntaxError
import logging
import shutil
from collections import defaultdict

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)
pd.options.mode.chained_assignment = None

# Setup directories
BASE_DIR = "zba_financial_data"
ORIGINAL_DIR = os.path.join(BASE_DIR, "original")
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Create subdirectories for categorized files
for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"📁 Created directory: {os.path.join(CURRENT_DIR, subfolder)}")

for dir_path in [BASE_DIR, ORIGINAL_DIR, OUTPUT_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created directory: {dir_path}")

# Log file for tracking processed files
PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")

bank_url = 'https://www.ziraatbank-kosova.com/sq/pasqyrat-financiare'
print(f"🔗 Target URL: {bank_url}")

def load_processed_files():
    """Load list of already processed files from CSV"""
    if os.path.exists(PROCESSED_FILES_LOG):
        df = pd.read_csv(PROCESSED_FILES_LOG)
        return set(df['file_name'].tolist())
    return set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    """Save processed file to log"""
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({
            'file_name': [file_name],
            'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            'status': [status],
            'reporting_date': [reporting_date],
            'current_name': [current_name],
            'is_corrupt': [is_corrupt]
        })
        if os.path.exists(PROCESSED_FILES_LOG):
            existing = pd.read_csv(PROCESSED_FILES_LOG)
            df = pd.concat([existing, df], ignore_index=True)
        df.to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    """Scrape PDF links from the Ziraat page"""
    print(f"🔍 Attempting to connect to: {url}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        print(f"✅ Successfully connected! Status code: {response.status_code}")
        
        soup = BeautifulSoup(response.text, 'html.parser')
        pdf_links = []
        
        for link in soup.find_all('a', href=True):
            href = link['href']
            # Keywords: ppf (balance sheet), bgj (balance sheet), pa (income statement)
            if href.lower().endswith('.pdf') and (('ppf' in href.lower() or 'bgj' in href.lower()) or 'pa' in href.lower()):
                full_url = urljoin(url, href)
                pdf_links.append(full_url)
                print(f"   📄 Found PDF: {os.path.basename(full_url)}")
        
        print(f"\n✅ Total PDFs found: {len(pdf_links)}")
        return pdf_links
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to website: {e}")
        return []

def download_pdf(url, local_dir):
    """Download PDF to local directory"""
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    
    print(f"   ⬇️ Downloading: {file_name}")
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        
        file_size = os.path.getsize(local_path) / 1024
        print(f"   ✅ Downloaded: {file_name} ({file_size:.1f} KB)")
        return local_path
    except Exception as e:
        print(f"   ❌ Failed to download {file_name}: {e}")
        return None

def extract_date(text, filename):
    """Extract a date string from text or filename"""
    # Try to find date pattern like 31.12.2023
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    # Try filename
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    
    return None

def download_all(force=False):
    """Download PDFs and categorize them locally"""
    processed = set() if force else load_processed_files()
    if force:
        print("⚠️ Force mode enabled: all files will be downloaded")
    
    print(f"🔍 Scraping PDF links from: {bank_url}")
    pdf_links = scrape_pdf_links(bank_url)
    
    if not pdf_links:
        print("❌ No PDF links found")
        return []
    
    new_files = []
    file_metadata = []
    
    # Keywords for categorization
    Balance_sheet = ['ppf', 'bgj']
    Income_Statement = ['pa']
    
    categories = {
        'balance-sheet': Balance_sheet,
        'income-statement': Income_Statement
    }

    print(f"\n📥 Starting download of {len(pdf_links)} PDFs...")
    
    for i, pdf_url in enumerate(pdf_links, 1):
        file_name = os.path.basename(pdf_url)
        print(f"\n[{i}/{len(pdf_links)}] Processing: {file_name}")
        
        # Skip if already processed
        if file_name in processed and not force:
            print(f"   ⏭️ File already processed. Skipping...")
            continue

        new_files.append(file_name)
        
        # Download to original directory
        file_path = download_pdf(pdf_url, ORIGINAL_DIR)
        if not file_path:
            continue

        # Try to read metadata creation date
        dt = None
        try:
            with open(file_path, 'rb') as f:
                pdf_reader = PdfReader(f)
                metadata = pdf_reader.metadata
                if metadata:
                    for k, v in metadata.items():
                        if "/CreationDate" in k or "/ModDate" in k:
                            date_str = re.search(r'D:(\d+)', str(v))
                            if date_str:
                                dt = datetime.strptime(date_str.group(1), '%Y%m%d%H%M%S')
                                print(f"   📅 PDF creation date: {dt.date()}")
                                break
        except Exception as e:
            print(f"   ⚠️ Could not read metadata: {e}")

        # Categorize based on filename
        file_lower = file_name.lower()
        found_category = None
        
        for category, keywords in categories.items():
            if any(keyword in file_lower for keyword in keywords):
                found_category = category
                break

        # Try to extract date from PDF content
        date = None
        try:
            text = extract_text(file_path)
            # Extract just the year (Ziraat uses year only)
            year_match = re.findall(r'\d{4}', text)
            if year_match:
                year = year_match[0]
                # For Ziraat, we'll use a placeholder date with the year
                # The actual quarter will be determined by page number during extraction
                date = f'31.12.{year}'  # Default to year-end
                print(f"   📅 Extracted year: {year}")
            else:
                date = extract_date(text, file_name)
                if date:
                    print(f"   📅 Extracted reporting date: {date}")
                else:
                    date = datetime.now().strftime('%d.%m.%Y')
                    print(f"   ⚠️ Could not extract date, using current: {date}")
        except Exception as e:
            print(f"   ⚠️ Error extracting text: {e}")
            date = datetime.now().strftime('%d.%m.%Y')

        if found_category:
            new_name = f"zba_{'bs' if found_category=='balance-sheet' else 'is'}_{date}.pdf"
            
            # Copy to categorized folder
            dest_dir = os.path.join(CURRENT_DIR, found_category)
            dest_path = os.path.join(dest_dir, new_name)
            shutil.copy2(file_path, dest_path)
            
            file_metadata.append({
                'file_name': file_name,
                'file_path': file_path,
                'current_name': new_name,
                'category': found_category,
                'reporting_date': date,
                'creation_date': dt,
                'download_date': datetime.now(),
                'is_corrupt': 0
            })
            
            print(f"   ✅ Categorized as {found_category} -> {new_name}")
        else:
            print(f"   ⚠️ Could not categorize")
            file_metadata.append({
                'file_name': file_name,
                'file_path': file_path,
                'current_name': None,
                'category': 'uncategorized',
                'reporting_date': date,
                'creation_date': dt,
                'download_date': datetime.now(),
                'is_corrupt': 0
            })
        
        # Mark as processed
        save_processed_file(file_name, reporting_date=date, current_name=new_name if found_category else None)

    # Save metadata
    if file_metadata:
        metadata_df = pd.DataFrame(file_metadata)
        metadata_path = os.path.join(OUTPUT_DIR, f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\n💾 File metadata saved to {metadata_path}")

    if new_files:
        print(f"\n✅ Downloaded {len(new_files)} new files.")
    else:
        print("\n✅ No new files detected.")

    return [m.get('current_name') for m in file_metadata if m.get('current_name')]

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

# Data storage
income_data = defaultdict(list)
balance_data = defaultdict(list)

def read_balance(pdf_path, filename):
    """Read balance sheet from Ziraat PDF (multiple pages for different quarters)"""
    file_current_name = os.path.basename(filename)
    
    try:
        # Extract year from PDF text
        text = extract_text(pdf_path)
        year_match = re.findall(r'\d{4}', text)
        year = year_match[0] if year_match else datetime.now().strftime('%Y')
        
        # Get number of pages
        with open(pdf_path, 'rb') as pdf_file:
            pdf_reader = PdfReader(pdf_file)
            num_pages = len(pdf_reader.pages)
        
        date_map = {
            1: '31.03.',
            2: '30.06.',
            3: '30.09.',
            4: '31.12.'
        }
        
        all_dfs = []
        
        for page_num in range(1, num_pages + 1):
            if page_num > 4:  # Ziraat typically has up to 4 quarters
                break
                
            dt_report = date_map[page_num] + year
            
            try:
                # Extract table from current page
                tables = cam.read_pdf(pdf_path, flavor='stream', pages=str(page_num), 
                                     table_areas=['0,792,612,0'], row_tol=16)
                
                dfs = []
                for table in tables:
                    df = table.df
                    dfs.append(df)
                
                if not dfs:
                    continue
                    
                full_df = pd.concat(dfs, ignore_index=True)
                
                # Clean data
                for col in [1, 2]:
                    full_df[col] = full_df[col].astype(str).str.replace(r'-(\s*)\n', '-\n', regex=True)
                    full_df[col] = full_df[col].str.replace(r'\n', '', regex=True)
                
                full_df = full_df[full_df[0] != '']
                full_df = full_df[(full_df[1] != '') & (full_df[2] != '')]
                
                if len(full_df.columns) >= 3:
                    full_df = full_df.iloc[:, :3]
                    full_df.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE']
                    
                    # Clean numeric values
                    for col in ['PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE']:
                        full_df[col] = full_df[col].astype(str).str.replace('-', '0', regex=False)
                        full_df[col] = full_df[col].str.replace(',', '')
                        full_df[col] = pd.to_numeric(full_df[col], errors='coerce').fillna(0)
                        full_df[col] = full_df[col].round().astype(int)
                    
                    full_df['DT_REPORT'] = dt_report
                    full_df['FILE_NAME'] = file_current_name
                    
                    all_dfs.append(full_df)
                    print(f"      ✅ Extracted balance sheet for {dt_report}")
                    
            except Exception as e:
                print(f"      ⚠️ Error on page {page_num}: {e}")
                continue
        
        if all_dfs:
            combined_df = pd.concat(all_dfs, ignore_index=True)
            return combined_df
        else:
            return pd.DataFrame()
            
    except Exception as e:
        print(f"      ❌ Error reading balance sheet: {e}")
        return pd.DataFrame()

def read_income(pdf_path, filename):
    """Read income statement from Ziraat PDF (multiple pages for different quarters)"""
    file_current_name = os.path.basename(filename)
    
    try:
        # Extract year from PDF text
        text = extract_text(pdf_path)
        year_match = re.findall(r'\d{4}', text)
        year = year_match[0] if year_match else datetime.now().strftime('%Y')
        
        # Get number of pages
        with open(pdf_path, 'rb') as pdf_file:
            pdf_reader = PdfReader(pdf_file)
            num_pages = len(pdf_reader.pages)
        
        date_map = {
            1: '31.03.',
            2: '30.06.',
            3: '30.09.',
            4: '31.12.'
        }
        
        all_dfs = []
        
        for page_num in range(1, num_pages + 1):
            if page_num > 4:  # Ziraat typically has up to 4 quarters
                break
                
            dt_report = date_map[page_num] + year
            
            try:
                # Extract table from current page
                tables = cam.read_pdf(pdf_path, flavor='stream', pages=str(page_num))
                
                dfs = []
                for table in tables:
                    df = table.df
                    dfs.append(df)
                
                if not dfs:
                    continue
                    
                full_df = pd.concat(dfs, ignore_index=True)
                
                # Clean data
                for col in [1, 2]:
                    full_df[col] = full_df[col].astype(str).str.replace(r'-(\s*)\n', '-\n', regex=True)
                    full_df[col] = full_df[col].str.replace(r'\n', '', regex=True)
                
                full_df = full_df[full_df[0] != '']
                full_df = full_df[(full_df[1] != '') & (full_df[2] != '')]
                
                if len(full_df.columns) >= 3:
                    full_df = full_df.iloc[:, :3]
                    full_df.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE']
                    
                    # Clean numeric values
                    for col in ['PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE']:
                        full_df[col] = full_df[col].astype(str).str.replace('-', '0', regex=False)
                        full_df[col] = full_df[col].str.replace(',', '')
                        full_df[col] = pd.to_numeric(full_df[col], errors='coerce').fillna(0)
                        full_df[col] = full_df[col].round().astype(int)
                    
                    full_df['DT_REPORT'] = dt_report
                    full_df['FILE_NAME'] = file_current_name
                    
                    all_dfs.append(full_df)
                    print(f"      ✅ Extracted income statement for {dt_report}")
                    
            except Exception as e:
                print(f"      ⚠️ Error on page {page_num}: {e}")
                continue
        
        if all_dfs:
            combined_df = pd.concat(all_dfs, ignore_index=True)
            return combined_df
        else:
            return pd.DataFrame()
            
    except Exception as e:
        print(f"      ❌ Error reading income statement: {e}")
        return pd.DataFrame()

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns"""
    for col in [prev_col, curr_col]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    """Main execution function that returns unified DataFrames"""
    
    print("\n" + "="*60)
    print("🏦 ZIRAAT BANK FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    # Clear previous data
    global income_data, balance_data
    income_data.clear()
    balance_data.clear()
    
    # Decide which files to process
    if extract_only:
        print("\n📂 EXTRACT-ONLY MODE: Processing existing files")
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            category_dir = os.path.join(CURRENT_DIR, category)
            if os.path.exists(category_dir):
                category_files = [f for f in os.listdir(category_dir) if f.startswith('zba_') and f.endswith('.pdf')]
                new_files.extend(category_files)
        print(f"Found {len(new_files)} existing files")
    else:
        print("\n🌐 DOWNLOAD MODE: Downloading and processing new files")
        new_files = download_all(force=force)
    
    if new_files or extract_only:
        # Process balance sheet files
        bs_dir = os.path.join(CURRENT_DIR, 'balance-sheet')
        if os.path.exists(bs_dir):
            print(f"\n📂 Processing balance sheet files...")
            for filename in os.listdir(bs_dir):
                if filename.startswith('zba_bs_') and filename.endswith('.pdf'):
                    if filename in new_files or extract_only:
                        file_path = os.path.join(bs_dir, filename)
                        print(f"\n   🔍 Processing: {filename}")
                        
                        # Read balance sheet
                        balance_df = read_balance(file_path, filename)
                        
                        # Store in balance_data
                        if not balance_df.empty:
                            for _, row in balance_df.iterrows():
                                date_obj = datetime.strptime(row['DT_REPORT'], '%d.%m.%Y')
                                date_formatted = date_obj.strftime('%Y-%m-%d')
                                
                                balance_data[date_formatted].append({
                                    'Category': row['BALANCE_CATEGORY'],
                                    'PREVIOUS_QUARTER': row['PREVIOUS_BALANCE_VALUE'],
                                    'CURRENT_QUARTER': row['CURRENT_BALANCE_VALUE'],
                                    'DT_REPORT': date_formatted,
                                    'FILE_NAME': filename
                                })
        
        # Process income statement files
        is_dir = os.path.join(CURRENT_DIR, 'income-statement')
        if os.path.exists(is_dir):
            print(f"\n📂 Processing income statement files...")
            for filename in os.listdir(is_dir):
                if filename.startswith('zba_is_') and filename.endswith('.pdf'):
                    if filename in new_files or extract_only:
                        file_path = os.path.join(is_dir, filename)
                        print(f"\n   🔍 Processing: {filename}")
                        
                        # Read income statement
                        income_df = read_income(file_path, filename)
                        
                        # Store in income_data
                        if not income_df.empty:
                            for _, row in income_df.iterrows():
                                date_obj = datetime.strptime(row['DT_REPORT'], '%d.%m.%Y')
                                date_formatted = date_obj.strftime('%Y-%m-%d')
                                
                                income_data[date_formatted].append({
                                    'Category': row['INCOME_CATEGORY'],
                                    'PREVIOUS_QUARTER': row['PREVIOUS_INCOME_VALUE'],
                                    'CURRENT_QUARTER': row['CURRENT_INCOME_VALUE'],
                                    'DT_REPORT': date_formatted,
                                    'FILE_NAME': filename
                                })
        
        # Create DataFrames
        print("\n📊 Creating DataFrames...")
        
        # Balance Sheet DataFrame
        full_bs = pd.DataFrame()
        for date, data in balance_data.items():
            df = pd.DataFrame(data)
            full_bs = pd.concat([full_bs, df], ignore_index=True)
        
        if not full_bs.empty:
            full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
            
            # Category mapping for balance sheet
            bs_map = {
                'Paraja e gatshme dhe gjendja me BQK-në': '20',
                'Kërkesat ndaj bankave': '21',
                'Bonot e thesarit': '22',
                'Investimet në letra me vlerë': '23',
                'Plasmanet ne Institucionet kreditore': '25',
                'Kreditë dhe paradhëniet ndaj klientëve': '26',
                'Patundshmëritë dhe pajisjet': '27',
                'Pasuritë e paprekshme': '28',
                'Pasuritë tatimore të shtyra': '29',
                'Pasuritë tjera': '30',
                'Gjithsej pasuritë': '31',
                'Depozitat e klientëve': '32',
                'Detyrimet ndaj bankave': '33',
                'Fondet tjera të huazuara': '34',
                'Detyrimet tatimore të shtyra': '35',
                'Detyrimet tjera': '36',
                'Gjithsej detyrimet': '37',
                'Kapitali aksionar': '38',
                'Rezervat e kapitalit': '40',
                'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
                'Fitimi i mbajtur': '41',
                'Fitimi/(humbja) e vitit aktual': '42',
                'Fitimi/(humbja) i vitit aktual': '42',
                'Përbërësit tjerë të ekuitetit': '43',
                'Përbërësit tjerë të ekuititetit': '43',
                'Gjithsej ekuiteti i aksionarëve': '44',
                'Gjithsej ekuititeti i aksionarëve': '44',
                'Gjithsej detyrimet dhe ekuiteti i \naksionarëve': '45',
                'Gjithsej detyrimet dhe ekuititeti i \naksionarëve': '45',
                'Gjithsej detyrimet dhe ekuiteti i aksionarëve': '45',
                'Gjithsej ekuiteti dhe detyrimet': '45',
                'aksionarëve': '45'
            }
            
            full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
            full_bs['BANK_ID'] = 9  # Ziraat Bank ID
            full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
            full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'])
        
        # Income Statement DataFrame
        full_is = pd.DataFrame()
        for date, data in income_data.items():
            df = pd.DataFrame(data)
            full_is = pd.concat([full_is, df], ignore_index=True)
        
        if not full_is.empty:
            full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
            
            # Category mapping for income statement
            is_map = {
                'Të hyrat nga interesi': '1',
                'Shpenzimet e interesit': '2',
                'Neto të hyrat nga interesi': '3',
                'Të hyrat nga tarifat dhe komisionet': '4',
                'Shpenzimet e tarifave dhe komisioneve': '5',
                'Neto të hyrat nga tarifat dhe komisionet': '6',
                'Neto të hyrat nga tregtimi': '7',
                'Neto të hyrat nga instrumentet tjera financiare': '8',
                'Neto të hyrat (shpenzimet) tjera operative': '9',
                'Neto të hyrat tjera': '80',
                'Gjithsej të hyrat': '10',
                'Provizionet për humbjet nga kreditë': '12',
                'Provizionet tjera': '13',
                'Fitimi/(humbja) para tatimit': '14',
                'Shpenzimet e tatimit në fitim': '15',
                'Shpenzimet e tatimit të shtyer': '15',
                'Fitimi/(humbja) neto': '16',
                'Të ardhurat tjera gjithëpërfshirëse': '17',
                'Gjithsej të ardhurat gjithëpërfshirëse': '19'
            }
            
            full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
            full_is['BANK_ID'] = 9  # Ziraat Bank ID
            full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
            full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'])
        
        # Create unified DataFrame
        unified_dfs = []
        
        if not full_is.empty:
            is_unified = full_is.rename(columns={
                'INCOME_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'
            })
            is_unified['CATEGORY_TYPE'] = 'INCOME'
            unified_dfs.append(is_unified)
        
        if not full_bs.empty:
            bs_unified = full_bs.rename(columns={
                'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'
            })
            bs_unified['CATEGORY_TYPE'] = 'BALANCE'
            unified_dfs.append(bs_unified)
        
        if unified_dfs:
            final_df = pd.concat(unified_dfs, ignore_index=True)
            final_df['CURR_ID'] = 1
            final_df['DATA_SOURCE'] = 'Ziraat Bank'
            final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
            final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
            
            # Drop rows with invalid dates or categories
            final_df = final_df.dropna(subset=['DT_REPORT', 'CATEGORY_CODE'])
            
            # Sort
            final_df = final_df.sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
            
            # Reorder columns
            column_order = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE',
                          'CATEGORY_CODE', 'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE',
                          'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
            
            final_columns = [col for col in column_order if col in final_df.columns]
            final_df = final_df[final_columns]
            
            # Save outputs
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Save unified DataFrame
            unified_path = os.path.join(OUTPUT_DIR, f"zba_unified_financial_data_{timestamp}.csv")
            final_df.to_csv(unified_path, index=False)
            print(f"\n💾 Unified data saved to: {unified_path}")
            
            # Save individual files
            if not full_is.empty:
                is_path = os.path.join(OUTPUT_DIR, f"zba_income_statement_{timestamp}.csv")
                full_is.to_csv(is_path, index=False)
            
            if not full_bs.empty:
                bs_path = os.path.join(OUTPUT_DIR, f"zba_balance_sheet_{timestamp}.csv")
                full_bs.to_csv(bs_path, index=False)
            
            # Save Excel with multiple sheets
            excel_path = os.path.join(OUTPUT_DIR, f"zba_financial_report_{timestamp}.xlsx")
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Unified_Data', index=False)
                if not full_is.empty:
                    full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
                if not full_bs.empty:
                    full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
            
            print(f"💾 Excel report saved to: {excel_path}")
            
            print(f"\n📊 UNIFIED DATAFRAME CREATED:")
            print(f"   - Total rows: {len(final_df)}")
            print(f"   - Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
            print(f"   - Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")
            print(f"   - Unique report dates: {final_df['REPORT_DATE'].nunique()}")
            print(f"   - Date range: {final_df['REPORT_DATE'].min()} to {final_df['REPORT_DATE'].max()}")
            
            elapsed_time = time.time() - start
            print(f"\n⏱️ Processing completed in {elapsed_time:.2f} seconds")
            
            return final_df, full_is, full_bs
        
        return pd.DataFrame(), full_is, full_bs
    
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Run the extraction
print("🚀 Starting Ziraat Bank financial data extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print("📊 FINAL UNIFIED DATAFRAME")
    print("="*60)
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    
    print("\n📊 Summary by Statement Type:")
    print(final_df['STATEMENT_TYPE'].value_counts())
    
    print("\n📅 Reports by Date:")
    print(final_df['REPORT_DATE'].value_counts().sort_index())
    
    print("\n📊 Data Types:")
    print(final_df.dtypes)
else:
    print("❌ No data in final DataFrame")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


📁 Created directory: zba_financial_data/current/balance-sheet
📁 Created directory: zba_financial_data/current/income-statement
📁 Created directory: zba_financial_data
📁 Created directory: zba_financial_data/original
📁 Created directory: zba_financial_data/output
🔗 Target URL: https://www.ziraatbank-kosova.com/sq/pasqyrat-financiare
🚀 Starting Ziraat Bank financial data extraction...

🏦 ZIRAAT BANK FINANCIAL DATA EXTRACTION

🌐 DOWNLOAD MODE: Downloading and processing new files
⚠️ Force mode enabled: all files will be downloaded
🔍 Scraping PDF links from: https://www.ziraatbank-kosova.com/sq/pasqyrat-financiare
🔍 Attempting to connect to: https://www.ziraatbank-kosova.com/sq/pasqyrat-financiare
✅ Successfully connected! Status code: 200
   📄 Found PDF: bgj-2025-tm1234.pdf
   📄 Found PDF: zfg_kosovo_bgj-2024-tm1234-pdf_12.pdf
   📄 Found PDF: zfg_kosovo_bgj-2023-tm1234-pdf_839.pdf
   📄 Found PDF: zfg_kosovo_bgj2022-pdf_760.pdf
   📄 Found PDF: zfg_kosovo_ppf2021tm1234-pdf_509.pdf
   📄 Foun

2026-03-18T15:35:16 - INFO - Processing page-1
2026-03-18 15:35:16,732 - INFO - Processing page-1
2026-03-18T15:35:16 - INFO - Processing page-2
2026-03-18 15:35:16,802 - INFO - Processing page-2


      ✅ Extracted balance sheet for 31.03.2015
      ✅ Extracted balance sheet for 30.06.2015

   🔍 Processing: zba_bs_31.12.2017.pdf


2026-03-18T15:35:16 - INFO - Processing page-1
2026-03-18 15:35:16,953 - INFO - Processing page-1
2026-03-18T15:35:17 - INFO - Processing page-2
2026-03-18 15:35:17,010 - INFO - Processing page-2
2026-03-18T15:35:17 - INFO - Processing page-3
2026-03-18 15:35:17,067 - INFO - Processing page-3
2026-03-18T15:35:17 - INFO - Processing page-4
2026-03-18 15:35:17,125 - INFO - Processing page-4


...

      ✅ Extracted income statement for 31.12.2022

   🔍 Processing: zba_is_31.12.2015.pdf
      ✅ Extracted income statement for 31.03.2015
      ✅ Extracted income statement for 30.06.2015

   🔍 Processing: zba_is_31.12.2016.pdf


2026-03-18T15:35:24 - INFO - Processing page-1
2026-03-18 15:35:24,178 - INFO - Processing page-1
2026-03-18T15:35:24 - INFO - Processing page-2
2026-03-18 15:35:24,237 - INFO - Processing page-2
2026-03-18T15:35:24 - INFO - Processing page-3
2026-03-18 15:35:24,295 - INFO - Processing page-3
2026-03-18T15:35:24 - INFO - Processing page-4
2026-03-18 15:35:24,355 - INFO - Processing page-4


      ✅ Extracted income statement for 31.03.2016
      ✅ Extracted income statement for 30.06.2016
      ✅ Extracted income statement for 30.09.2016
      ✅ Extracted income statement for 31.12.2016

   🔍 Processing: zba_is_31.12.2017.pdf


2026-03-18T15:35:24 - INFO - Processing page-1
2026-03-18 15:35:24,501 - INFO - Processing page-1
2026-03-18T15:35:24 - INFO - Processing page-2
2026-03-18 15:35:24,564 - INFO - Processing page-2
2026-03-18T15:35:24 - INFO - Processing page-3
2026-03-18 15:35:24,627 - INFO - Processing page-3
2026-03-18T15:35:24 - INFO - Processing page-4
2026-03-18 15:35:24,690 - INFO - Processing page-4


      ✅ Extracted income statement for 31.03.2017
      ✅ Extracted income statement for 30.06.2017
      ✅ Extracted income statement for 30.09.2017
      ✅ Extracted income statement for 31.12.2017

📊 Creating DataFrames...

💾 Unified data saved to: zba_financial_data/output/zba_unified_financial_data_20260318_153524.csv
💾 Excel report saved to: zba_financial_data/output/zba_financial_report_20260318_153524.xlsx

📊 UNIFIED DATAFRAME CREATED:
   - Total rows: 1376
   - Income Statement rows: 600
   - Balance Sheet rows: 776
   - Unique report dates: 42
   - Date range: 2015-03-31 to 2025-12-31

⏱️ Processing completed in 91.91 seconds

📊 FINAL UNIFIED DATAFRAME
Shape: (1376, 13)

First 10 rows:
     BANK_ID REPORT_DATE  DT_REPORT STATEMENT_TYPE CATEGORY_TYPE  \
604        9  2015-03-31 2015-03-31  BALANCE_SHEET       BALANCE   
605        9  2015-03-31 2015-03-31  BALANCE_SHEET       BALANCE   
606        9  2015-03-31 2015-03-31  BALANCE_SHEET       BALANCE   
607        9  2015-03-3

In [3]:
final_df.to_csv("ZBA.csv")

In [4]:
final_df

,BANK_ID,REPORT_DATE,DT_REPORT,STATEMENT_TYPE,CATEGORY_TYPE,CATEGORY_CODE,ORIGINAL_CATEGORY,PREVIOUS_VALUE,CURRENT_VALUE,CURR_ID,FILE_NAME,DATA_SOURCE,EXTRACTION_DATE
604,9,2015-03-31,2015-03-31,BALANCE_SHEET,BALANCE,20,Paraja e gatshme dhe gjendja me BQK-në,10,8,1,zba_bs_31.12.2015.pdf,Ziraat Bank,2026-03-18
605,9,2015-03-31,2015-03-31,BALANCE_SHEET,BALANCE,21,Kërkesat ndaj bankave,55,102,1,zba_bs_31.12.2015.pdf,Ziraat Bank,2026-03-18
606,9,2015-03-31,2015-03-31,BALANCE_SHEET,BALANCE,22,Bonot e thesarit,0,0,1,zba_bs_31.12.2015.pdf,Ziraat Bank,2026-03-18
607,9,2015-03-31,2015-03-31,BALANCE_SHEET,BALANCE,23,Investimet në letra me vlerë,0,0,1,zba_bs_31.12.2015.pdf,Ziraat Bank,2026-03-18
608,9,2015-03-31,2015-03-31,BALANCE_SHEET,BALANCE,26,Kreditë dhe paradhëniet ndaj klientëve,0,978,1,zba_bs_31.12.2015.pdf,Ziraat Bank,2026-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,9,2025-12-31,2025-12-31,INCOME_STATEMENT,INCOME,4,Të hyrat nga tarifat dhe komisionet,479,87,1,zba_is_31.12.2025.pdf,Ziraat Bank,2026-03-18
59,9,2025-12-31,2025-12-31,INCOME_STATEMENT,INCOME,5,Shpenzimet e tarifave dhe komisioneve,353,98,1,zba_is_31.12.2025.pdf,Ziraat Bank,2026-03-18
60,9,2025-12-31,2025-12-31,INCOME_STATEMENT,INCOME,6,Neto të hyrat nga tarifat dhe komisionet,126,11,1,zba_is_31.12.2025.pdf,Ziraat Bank,2026-03-18
61,9,2025-12-31,2025-12-31,INCOME_STATEMENT,INCOME,80,Neto të hyrat tjera,150,18,1,zba_is_31.12.2025.pdf,Ziraat Bank,2026-03-18
